In [ ]:
import os, sys, glob
sys.path.insert(0, os.path.abspath('.'))  # self-contained: DescriptorDOS/ copied into this folder

from mpi4py import MPI
from DescriptorDOS import DDOSManager

import numpy as np
import matplotlib.pyplot as plt

## Sampling the liquid phase

This notebook studies the **liquid** phase of W, sampled by NVE around the
`W-liquid.dat` configuration (128 atoms). Unlike the solid bcc/a15 phases,
the liquid has no analytic harmonic (Hessian) reference, so it is the more
fragile case for the DescriptorDOS method: the sampling potential, the
reference free energy, and the linear (SNAP) model being scored are three
separate choices that all have to be tracked and made consistent. The goal
here is to lay out, step by step, where each of those choices can go wrong.

We sample under the analytic Uhlenbeck-Ford (UF) reference fluid
(`pair_style ufm`), which is a well-known reference with an analytic free energy
 in this codebase (see `ufGenerator.py`). `p` must be
one of `{1,25,50,75,100}` - the only values tabulated there - and `epsilon
= p * kB * T_ref` is fixed once for the whole run, not recomputed per
target temperature.

In [ ]:
ref_path = "W/"

# Uhlenbeck-Ford reference fluid driving the NVE dynamics.
UF_p, UF_sigma, T_ref = 50, 1.2, 3800.0
kB = 8.617333262e-5  # eV/K
epsilon = UF_p * kB * T_ref

mass = 183.84
strain_list = [0.0]
# liquid W is stable around 3500K-4500K, so sample close to that range
isosurface_temperature_list = list(np.linspace(3.0, 4.5, 15))

DDOS_Manager = DDOSManager(
    comm=MPI.COMM_WORLD,

    # the only reference fluid with a known analytic free energy here
    isosurface='UhlenbeckFord',

    # extra configuration options (mostly overwritten below)
    yaml_file="configuration-fine.yaml",

    # LAMMPS input script: pair_style ufm + the real SNAP descriptor compute
    lammps_input_script=os.path.join(ref_path, "in_uf.lammps"),

    # reuse the dataset already sampled (run.py --structure liquid --potential uf, see examples/)
    dump_path=os.path.join(ref_path, "output/"),
    dump_file=f"DDOSData-W-liquid-UhlenbeckFord-uf_p{UF_p}_s{UF_sigma}",

    # number of descriptor samples per isosurface value
    default_calls=4000,

    # path and filename for the liquid input structure - already at the target density
    input_path=os.path.join(ref_path, 'configurations/'),
    input_data='W-liquid.dat',

    # path and filename for Hessian data (unused for 'UhlenbeckFord', kept for interface consistency)
    hessian_path=os.path.join(ref_path, 'hessians/'),
    hessian_data='W-liquid.pkl',

    verbose=False,
    store_uncompressed_data=False,
    percentile_threshold=5,

    input_parameters={'IsoSurface': isosurface_temperature_list, 'Strain': strain_list},
    compute_covar=False,

    # if True, will append data to existing file rather than overwriting matching jobs
    append_data=False,

    # pair_coeff for pair_style ufm: "epsilon sigma", not a file
    pair_coeff=f"{epsilon:.6f} {UF_sigma}",
    mass=mass,

    # passed through to UhlenbeckFordDisplacer so its bookkeeping (used when
    # fitting the reference free energy) matches what is actually run here
    UF_p=UF_p,
    UF_sigma=UF_sigma,
)



In [ ]:
DDOS_Manager.run()

## Reference free energy: ideal gas -> Uhlenbeck-Ford

Before we can score the SNAP model against this dataset, `DDOSAnalyzer`
needs the **reference free energy** `S_alpha(alpha, T)`: the entropy of
the biased ensemble that was actually sampled, as a function of the
isosurface variable `alpha`. For the Hessian displacer this is known
analytically (harmonic crystal); for the UF reference fluid it is built
in two pieces, following Paula Leite & de Koning:

$$F_{UF}(T) = F_{\text{ideal gas}}(T) + \Delta F_{\text{ideal gas} \to UF}(T)$$

- $F_{\text{ideal gas}}(T)$ is the classical ideal-gas free energy at the
  sampled density (closed form, de Broglie thermal wavelength).
- $\Delta F_{\text{ideal gas} \to UF}(T)$ is the excess free energy of
  turning on the UF pair interaction, read off the tabulated virial-series
  splines in `ufGenerator.py` for the `p` actually used to sample (must be
  one of `{1,25,50,75,100}`).

We evaluate this sum on the sampled temperature range to get `F_ref(T)`,
then call `analyzer.fit_S_alpha(Beta, F_ref)`, which fits `S_alpha` the reference entropy

In [ ]:
import pickle as pkl
import ufGenerator as uf
from DescriptorDOS import DDOSAnalyzer
from DescriptorDOS.Constants import hbar, eV, atomic_mass


def F_UF(beta_array, volume_per_atom, mass, sigma, p):
    """ F_UF/N = F_ideal_gas/N + F_(ideal_gas->UF)/N (Paula Leite & de Koning virial series). """
    if p not in uf.splines:
        raise ValueError(f"ufGenerator.py only tabulates p in {sorted(uf.splines)}, got p={p}")

    rho = 1.0 / volume_per_atom
    x = 0.5 * (np.pi * sigma**2)**1.5 * rho

    if x < 0.1:
        index = int(x * 400)
    elif x < 1:
        index = 40 + int(x * 40 - 4)
    elif x < 4:
        index = 76 + int(x * 10 - 10)
    else:
        index = 105

    coef = uf.splines[p][index]
    sum_spline = {1: uf.sum_spline1, 25: uf.sum_spline25, 50: uf.sum_spline50,
                  75: uf.sum_spline75, 100: uf.sum_spline100}[p]
    beta_Fex_N = uf.fe(x, coef, sum_spline, index)  # beta * F_(ideal_gas->UF) / N, adimensional

    prefactor = (2.0 * np.pi * hbar**2 / (atomic_mass * eV)) * 1e20
    Lambda = np.sqrt(prefactor * beta_array / mass)
    F_ig_N = (1.0 / beta_array) * (np.log(rho) - 1.0) + 3.0 / beta_array * np.log(Lambda)

    return F_ig_N + beta_Fex_N / beta_array


# Load the sampled dataset into an analyzer (same file the manager above just read)
with open(DDOS_Manager.Agent.file_name, "rb") as f:
    raw = pkl.load(f)
V0 = raw[0]["V_0"] / raw[0]["N"]

analyzer = DDOSAnalyzer(DDOS_Manager.Agent.file_name, volume_per_atom=V0, mass=mass, verbose=True)
assert analyzer.flavor == "uhlenbeckford"

# Reference free energy over the sampled temperature range
T_min, T_max = analyzer.T_Equipartition.min(), analyzer.T_Equipartition.max()
Beta_fit = 1.0 / (8.617333262145e-5 * np.linspace(T_min, T_max, 30))
F_ref = F_UF(Beta_fit, V0, mass, sigma=UF_sigma, p=UF_p)

# Fit S_alpha(alpha, T) so it reproduces F_ref at alpha -> 0 - nothing scored yet
analyzer.fit_S_alpha(Beta_fit, F_ref)

## Scoring the SNAP linear model: test vs. real potential

With `S_alpha` fitted above, we can now evaluate the **absolute** free
energy of any linear-in-descriptor (SNAP-like) model on this dataset,
without any further LAMMPS run: `cohesive_energy()` and
`score_free_energy()` only need `Theta` (the SNAP coefficients) and the
descriptor histogram already stored in the pickle.

We compare two SNAP potentials for W:
- **test**: `X_W.snapcoeff`, placeholder/dummy coefficients (beta0 = 0),
  used so far only to exercise the pipeline.
- **real**: `W_2940_2017_2.snapcoeff`, the published Wood & Thompson
  (2017) fit for W.

Each `.snapcoeff` file stores one constant offset `beta0` followed by the
`ncoeff` bispectrum coefficients `Theta`; the total cohesive energy is
`beta0 + Theta @ D`, not just `Theta @ D`.

In [ ]:
def load_snap_coeffs(path):
    """ Robust parse of a single-element LAMMPS .snapcoeff file -> (beta0, Theta[1,ncoeff]). """
    with open(path) as f:
        lines = [l.strip() for l in f if l.strip() and not l.strip().startswith('#')]
    nelem, ncoeffall = (int(v) for v in lines[0].split())
    assert nelem == 1, "only single-element files are supported here"
    coeffs = np.array([float(v) for v in lines[2:2 + ncoeffall]])
    assert coeffs.size == ncoeffall
    return coeffs[0], coeffs[1:].reshape(1, -1)


potentials = {
    "test (X_W, dummy coefficients)": os.path.join(ref_path, "X_W.snapcoeff"),
    "real (Wood & Thompson 2017)": os.path.join(ref_path, "W_2940_2017_2.snapcoeff"),
}

Temperatures = np.linspace(T_min, T_max, 16)
results = {}

for label, path in potentials.items():
    beta0, Theta = load_snap_coeffs(path)
    analyzer.load_prediction(Theta, T=np.array([0.0]))

    E0_total = beta0 + analyzer.cohesive_energy(Thetas=analyzer.Thetas_raw)[0]

    F, errF, unsolved = [], [], []
    for T in Temperatures:
        result = analyzer.score_free_energy(Temperature=T)
        F.append(E0_total + result["F"][0])
        errF.append(result["errorF"][0])
        unsolved.append(bool(result["unsolved_models"][0]))

    results[label] = {"E0": E0_total, "T": Temperatures,
                       "F": np.array(F), "errF": np.array(errF), "unsolved": np.array(unsolved)}

    print(f"\n{label}: E(T=0) = {E0_total:.4f} eV/atom, "
          f"{sum(unsolved)}/{len(unsolved)} unsolved")
    for T, f, e, u in zip(Temperatures, F, errF, unsolved):
        print(f"  T={T:8.1f}K  F={f:10.4f} +/- {e:.4f} eV/atom" + ("  (unsolved)" if u else ""))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ref_label = next(iter(results))  # first potential = reference for the rescaled panel
E0_ref = results[ref_label]["E0"]

for label, res in results.items():
    line = axes[0].errorbar(res["T"], res["F"], yerr=res["errF"], marker='o', capsize=3, label=label)
    axes[0].axhline(res["E0"], linestyle='--', color=line[0].get_color(), alpha=0.6)

    # shift this curve so its T=0K offset matches the reference potential's -
    # makes the thermal *variation* directly comparable, independent of the
    # (much larger) static cohesive-energy offset between the two potentials
    shift = E0_ref - res["E0"]
    line2 = axes[1].errorbar(res["T"], res["F"] + shift, yerr=res["errF"], marker='o', capsize=3, label=label)
    axes[1].axhline(E0_ref, linestyle='--', color=line2[0].get_color(), alpha=0.6)

axes[0].plot([], [], linestyle='--', color='gray', label='E(0K)')
axes[0].set_xlabel("T [K]")
axes[0].set_ylabel("F_liquid [eV/atom]")
axes[0].set_title(f"Absolute (UF reference: p={UF_p}, sigma={UF_sigma})")
axes[0].legend()

axes[1].set_xlabel("T [K]")
axes[1].set_ylabel("F_liquid [eV/atom] (rescaled)")
axes[1].set_title(f"Rescaled to {ref_label}'s E(0K) - thermal variation only")
axes[1].legend()

plt.tight_layout()
plt.show()

## Major problem: the absolute free energy depends on (p, sigma)

`p` and `sigma` are not measured or fitted to anything physical here - they
are free parameters of the *reference* fluid we chose to sample under, and
`fit_S_alpha` reproduces a different `F_ref(T)` for each choice. Since the
absolute free energy reported above is `F(T) = F_ref(T) + deltaF(T)`, any
sensitivity of `F_ref` to (p, sigma) leaks directly into the supposedly
absolute SNAP free energy, even though the dynamics, the descriptor, and
the SNAP coefficients are unchanged - only the bookkeeping reference fluid
differs.

We check this directly: every UF-sampled pickle in `W/output/` (each a
different (p, sigma) pair, same liquid configuration) is independently
fitted and scored against the **real** SNAP potential. If the method gave
a genuinely absolute free energy, all the curves below would overlap -
they don't.

In [ ]:
beta0_real, Theta_real = load_snap_coeffs(os.path.join(ref_path, "W_2940_2017_2.snapcoeff"))

pkl_files = sorted(glob.glob(os.path.join(ref_path, "output", "DDOSData-W-liquid-UhlenbeckFord-*-UhlenbeckFord.pkl")))
print(f"Found {len(pkl_files)} UF-sampled pickle(s):")
for path in pkl_files:
    print(" ", path)

plt.figure(figsize=(6.5, 5))

for path in pkl_files:
    with open(path, "rb") as f:
        raw_i = pkl.load(f)
    V0_i = raw_i[0]["V_0"] / raw_i[0]["N"]
    p_i, sigma_i = raw_i[0]["p"], raw_i[0]["sigma"]

    analyzer_i = DDOSAnalyzer(path, volume_per_atom=V0_i, mass=mass, verbose=False)
    assert analyzer_i.flavor == "uhlenbeckford"

    T_min_i, T_max_i = analyzer_i.T_Equipartition.min(), analyzer_i.T_Equipartition.max()
    Beta_fit_i = 1.0 / (8.617333262145e-5 * np.linspace(T_min_i, T_max_i, 30))
    F_ref_i = F_UF(Beta_fit_i, V0_i, mass, sigma=sigma_i, p=p_i)
    analyzer_i.fit_S_alpha(Beta_fit_i, F_ref_i)

    analyzer_i.load_prediction(Theta_real, T=np.array([0.0]))
    E0_i = beta0_real + analyzer_i.cohesive_energy(Thetas=analyzer_i.Thetas_raw)[0]

    Temperatures_i = np.linspace(T_min_i, T_max_i, 16)
    F_i, errF_i = [], []
    for T in Temperatures_i:
        result = analyzer_i.score_free_energy(Temperature=T)
        F_i.append(E0_i + result["F"][0])
        errF_i.append(result["errorF"][0])

    line = plt.errorbar(Temperatures_i, F_i, yerr=errF_i, marker='o', capsize=3, label=f"p={p_i}, sigma={sigma_i}")
    plt.axhline(E0_i, linestyle='--', color=line[0].get_color(), alpha=0.5)

plt.plot([], [], linestyle='--', color='gray', label='E(0K)')
plt.xlabel("T [K]")
plt.ylabel("F_liquid [eV/atom]")
plt.title("Real SNAP potential, scored against different UF references")
plt.legend()
plt.tight_layout()
plt.show()

## Next step: descriptor density along NVE trajectories

So far we only ever looked at *compressed* histograms produced by
`DDOSManager` (rank-one/histogram compression of the descriptor
distribution). As a more direct check, we now generate full NVE
trajectories and dump the real per-atom SNAP descriptor (the real Wood &
Thompson 2017 fit: `rcutfac=4.73442`, same compute as `W/in_uf.lammps`)
at every saved frame, under two different driving dynamics:

- **UF**: NVE under the analytic Uhlenbeck-Ford reference fluid
  (`pair_style ufm`, same `epsilon`/`sigma` as the sampling above).
- **SNAP**: NVE under the real SNAP potential itself.

The goal (next cells) is to compare the *sampled descriptor density*
produced by each dynamics - this is the distribution DescriptorDOS's
score-matching machinery implicitly assumes is well covered by the
reference (UF) sampling. For now we only generate the raw data: one
trajectory + descriptor dump and one energy log per dynamics, written to
`W/configurations/`. No analysis yet.

In [ ]:
from lammps import lammps

RUN_STEPS = 15000
DUMP_EVERY = 10
T_md = 3800.0

# descriptor compute always matches the real SNAP fit, regardless of which
# potential drives the dynamics - same convention as W/in_uf.lammps
rcutfac, rfac0, twojmax = 4.73442, 0.99363, 8
desc_kwargs = "rmin0 0 bzeroflag 0 quadraticflag 0"
R_W, w_W = 0.5, 1.0
desc_columns = " ".join(f"c_desc[{i+1}]" for i in range(55))

configurations_dir = os.path.join(ref_path, "configurations")
os.makedirs(configurations_dir, exist_ok=True)

for potential in ["uf", "snap"]:
    trajectory_file = os.path.join(configurations_dir, f"trajectory_NVE_{potential}.lammpstrj")
    energies_file = os.path.join(configurations_dir, f"energies_NVE_{potential}.txt")

    if potential == "uf":
        pair_block = f"pair_style ufm 10.0\npair_coeff * * {epsilon:.6f} {UF_sigma}"
    else:
        snapcoeff = os.path.join(ref_path, "W_2940_2017_2.snapcoeff")
        snapparam = os.path.join(ref_path, "W_2940_2017_2.snapparam")
        pair_block = f"pair_style snap\npair_coeff * * {snapcoeff} {snapparam} W"

    lmp = lammps(cmdargs=["-screen", "none"], comm=MPI.COMM_WORLD)
    lmp.commands_string(f"""
units           metal
atom_style      atomic
boundary        p p p
read_data       {os.path.join(configurations_dir, "W-liquid.dat")}
mass            * {mass}
{pair_block}
velocity        all create {T_md} 12345 dist gaussian
fix             1 all nve

compute         desc all sna/atom {rcutfac} {rfac0} {twojmax} {R_W} {w_W} {desc_kwargs}
dump            1 all custom {DUMP_EVERY} {trajectory_file} id type x y z {desc_columns}

variable        step_val equal step
variable        pe_val equal pe
variable        ke_val equal ke
variable        te_val equal etotal
fix             2 all print {DUMP_EVERY} "${{step_val}} ${{pe_val}} ${{ke_val}} ${{te_val}}" file {energies_file} screen no title "# Step PotentialEnergy(eV) KineticEnergy(eV) TotalEnergy(eV)"

timestep        0.001
run             {RUN_STEPS}
""")
    lmp.close()
    print(f"[{potential}] trajectory -> {trajectory_file}")
    print(f"[{potential}] energies   -> {energies_file}")

## Comparing the sampled descriptor densities

We now compare the distribution of global descriptors actually visited
by the two trajectories above (UF-driven vs. SNAP-driven NVE), following
the procedure in `examples/compute_histogram_descriptors.py`: extract a
global descriptor per frame, decorrelate it with a PCA, and compare
per-component histograms between the two trajectories.

This is exactly the implicit assumption behind the score-matching
machinery used earlier (`DDOSAnalyzer`'s `cov_evec`/`cov_eval` rotation):
that the histogram density estimated from one sampling (here, biased by
UF) is representative of $p(\mathbf{D}|\alpha)$, the descriptor density at
fixed isosurface value, regardless of which dynamics produced it.
Plotting both densities side by side is the most direct way to see
whether that holds.

**Objects and spaces.** A configuration is the vector of the $N=128$
atomic positions, $\mathbf{X} \in \mathbb{R}^{3N}$ (one frame of the
trajectory). Each atom $i$ carries a per-atom bispectrum descriptor
$\mathbf{D}_i(\mathbf{X}) \in \mathbb{R}^{d}$ ($d=55$ here), and the
**global descriptor** of the frame is their average (same convention as
the `aveD` compute in the LAMMPS scripts):
$$\mathbf{D}(\mathbf{X}) = \frac{1}{N}\sum_{i=1}^{N} \mathbf{D}_i(\mathbf{X}) \in \mathbb{R}^{d}$$

**PCA / decorrelation.** Over the frames $\{\mathbf{X}_t\}$ of the
reference trajectory, with mean $\bar{\mathbf{D}} = \langle \mathbf{D}(\mathbf{X}_t) \rangle_t$,
diagonalize the $d\times d$ covariance matrix
$$\Sigma = \big\langle (\mathbf{D}-\bar{\mathbf{D}})(\mathbf{D}-\bar{\mathbf{D}})^T \big\rangle = W \Lambda W^T,$$
where $W \in \mathbb{R}^{d\times d}$ is orthogonal (its columns are the
eigenvectors of $\Sigma$) and $\Lambda$ is diagonal. Use the *same* $W$
(fitted once, on one trajectory) to decorrelate both datasets into
$d$ independent-ish components:
$$\mathbf{D}'(\mathbf{X}) = W^T \big(\mathbf{D}(\mathbf{X}) - \bar{\mathbf{D}}\big) \in \mathbb{R}^{d}$$

Each scalar component $D'_k$ can then be histogrammed independently to
estimate $p(D'_k\,|\,\alpha)$ for each trajectory, and the two empirical
densities compared quantitatively with the Kullback-Leibler divergence:
$$D_{KL}(P\|Q) = \sum_b P_b \ln\frac{P_b}{Q_b}$$

In [ ]:
def extract_global_descriptors(path, n_desc=55):
    """ Per-frame global descriptor D(X) = mean over atoms of the per-atom bispectrum (aveD convention)."""
    with open(path) as f:
        lines = f.read().split("\n")

    frames = []
    i, n_lines = 0, len(lines)
    while i < n_lines:
        if lines[i].startswith("ITEM: NUMBER OF ATOMS"):
            n_atoms = int(lines[i + 1])
            i += 2
        elif lines[i].startswith("ITEM: BOX BOUNDS"):
            i += 4
        elif lines[i].startswith("ITEM: ATOMS"):
            i += 1
            block = lines[i:i + n_atoms]
            data = np.array([l.split()[-n_desc:] for l in block], dtype=float)
            frames.append(data.mean(axis=0))
            i += n_atoms
        else:
            i += 1
    return np.array(frames)


D_uf = extract_global_descriptors(os.path.join(configurations_dir, "trajectory_NVE_uf.lammpstrj"))
D_snap = extract_global_descriptors(os.path.join(configurations_dir, "trajectory_NVE_snap.lammpstrj"))
print(f"D_uf: {D_uf.shape[0]} frames, D_snap: {D_snap.shape[0]} frames, d={D_uf.shape[1]}")

# PCA fitted on the UF (reference) trajectory only, then applied to both - see markdown above
D_bar = D_uf.mean(axis=0)
Sigma = np.cov((D_uf - D_bar).T)
Lambda, W = np.linalg.eigh(Sigma)
order = np.argsort(Lambda)[::-1]
Lambda, W = Lambda[order], W[:, order]

Dp_uf = (D_uf - D_bar) @ W
Dp_snap = (D_snap - D_bar) @ W

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))

# as many histograms as there are descriptors (d=55) - we only show 6 of them,
# the 6 highest-variance PCA components
pcs_to_show = [0, 1, 2, 3, 4, 5]

for ax, k in zip(axes.flat, pcs_to_show):
    lo = min(Dp_uf[:, k].min(), Dp_snap[:, k].min())
    hi = max(Dp_uf[:, k].max(), Dp_snap[:, k].max())
    bins = np.linspace(lo, hi, 100)

    ax.hist(Dp_uf[:, k], bins=bins, density=True, alpha=0.55, color="tab:blue", label="UF-driven")
    ax.hist(Dp_snap[:, k], bins=bins, density=True, alpha=0.55, color="tab:orange", label="SNAP-driven")

    var_frac = Lambda[k] / Lambda.sum()
    ax.set_title(f"PC{k+1} ({100*var_frac:.1f}% var)")
    ax.set_xlabel(rf"$D'_{{{k+1}}}$")
    ax.set_ylabel("density")

axes.flat[0].legend(fontsize=8)
fig.suptitle("Descriptor density: UF-driven vs SNAP-driven NVE (6 of 55 PCA components)")
plt.tight_layout()
plt.show()

## Conclusion

Two more targeted attempts were made to remove the (p, sigma) dependence
found above, neither succeeded:

- **Fitting (p, sigma) by minimizing the descriptor KL divergence**
  (`examples/optimize_uf_parameters.py`, differential evolution on the mean
  `D_KL` across all 55 descriptor components between the UF- and
  SNAP-driven trajectories): no (p, sigma) choice brought the UF-sampled
  descriptor density meaningfully closer to the SNAP-sampled one overall.
- **Fitting the UFM pair potential directly against SNAP energies**
  (`examples/fit_uf_parameters.py`, least-squares match of
  $U_{UF}(r)$ to SNAP-computed energies along the same trajectory): same
  outcome - no (p, sigma) reproduces the SNAP energetics closely enough to
  remove the sensitivity seen in the multi-pickle comparison above.

So, with the tools used so far, the absolute free energy of the liquid
stays reference-dependent regardless of how (p, sigma) are chosen.

Ideas not yet tried:
- Weight the KL/energy fit by $|\Theta_k|$ (the real SNAP coefficients)
  instead of treating all 55 components equally - only the
  heavily-weighted components actually matter for the scored free energy,
  and they might be the ones UF reproduces worst.
- Replace the analytic UF pair potential by a numerically tabulated one
  fit directly to the SNAP pair energy vs. $r$, removing the (p, sigma)
  functional-form constraint entirely instead of just retuning it.
- Skip the analytic reference altogether: integrate the *measured* SNAP
  caloric curve $E(T)$ directly (thermodynamic integration on the real
  trajectory, no UF at all) - this only gives $F(T)$ up to one unknown
  additive constant, but removes the (p, sigma) sensitivity by
  construction.
- Check whether this specific SNAP fit (2017, likely trained mostly on
  solid bcc data) is simply out of its training domain for the liquid,
  independently of which reference is used - e.g. by repeating this whole
  notebook with a SNAP potential explicitly fitted including liquid
  configurations.